# AXIS-Y 비건 콜라겐 아이 세럼 — '아마존 100개 제한 절단 오류' 보정 및 시계열 인사이트 분석

- **분석 대상:** `AXIS-Y Vegan Collagen Eye Serum (ASIN: B0CWPCX1D2)` 아마존 글로벌 고객 리뷰
- **핵심 문제 해결 (아마존 필터당 100개 제한 오류 보정):**
  1. **별점별 표집률 불균형 보정 (`Horvitz-Thompson Weight`, $W_s = N_s / n_s$):** 모집단(4,325건) 중 72%인 5★(3,114건)가 100개 제한에 걸려 4.01%(125건)만 수집된 반면, 2★(60건)·3★(94건)·4★(99건)는 100개 미만으로 **전수 수집(Uncensored Census)**됨 → 별점별 복원 승수($W_5=24.91, W_4=5.24, W_3=3.68, W_2=2.88, W_1=1.73$) 적용
  2. **5★ 시간축 우측 절단(Right-Truncation) 및 `Most recent` 25건 편중 보정:** 100개 상한으로 인해 2025년 8월 이전의 5★ 리뷰가 페이지 밖으로 밀려나 누락된 현상을 무절단 전수층(`1★~4★`)의 시계열 분포를 통해 사후 층화 복원(`recon_pop_total`, `trunc_corrected_rating`)
  3. **미국 우선 노출(US-First Pagination)로 인한 글로벌 5★ 0건 누락 보정:** 미국 리뷰가 앞의 100칸을 모두 채워 글로벌 5★가 0건 수집된 구조적 절단을 규명하고, 무절단 동등 계층(`1★~4★`) 기준으로 국가별 실제 만족도를 재평가

In [1]:
import pandas as pd
import numpy as np

df_clean = pd.read_csv('data/reviews_cleaned.csv')
daily_df = pd.read_csv('data/timeseries_daily.csv')
weekly_df = pd.read_csv('data/timeseries_weekly.csv')
monthly_df = pd.read_csv('data/timeseries_monthly.csv')

print(f"정제 완료 개별 리뷰: {df_clean.shape} (모집단 복원 가중치 합계: {df_clean['ht_expansion_weight'].sum():.1f}건)")
print(f"일별 시계열: {daily_df.shape} | 주별 시계열: {weekly_df.shape} | 월별 시계열: {monthly_df.shape}")

정제 완료 개별 리뷰: (478, 36) (모집단 복원 가중치 합계: 4325.0건)
일별 시계열: (324, 17) | 주별 시계열: (126, 12) | 월별 시계열: (26, 41)


## 1. 아마존 100개 제한(100-Cap) 및 미국 우선 노출 구조 검증
- `5★`: 미국 125건 / 글로벌 **0건** (미국 리뷰가 100개 제한을 모두 소진하여 글로벌 5★ 진입 불가)
- `4★ (99건)`, `3★ (94건)`, `2★ (60건)`: 100개 미만으로 미국 리뷰가 모두 끝난 뒤 글로벌 리뷰까지 **100% 전수 수집**

In [2]:
print(pd.crosstab(df_clean['rating_num'], df_clean['region_group'], margins=True))

region_group  Global (비미국 18개국)  US (미국)  All
rating_num                                   
1.0                          29       71  100
2.0                          22       38   60
3.0                          50       44   94
4.0                          51       48   99
5.0                           0      125  125
All                         152      326  478


## 2. 보정 전(표본 단순집계) vs 보정 후(절단 복원 모집단 집계) 월별 시계열 비교
- **유입량 피크 역전:** 보정 전에는 `2026-03`(43건)이 최대 피크처럼 보였으나, 이는 `Most recent` 25건이 5★에만 추가 수집된 착시임. 절단 보정 후 실제 최대 피크는 **`2025-11` 블랙프라이데이(345.5건)**, 2위는 **`2026-04`(299.8건)** 및 **`2026-07` 프라임데이(289.4건)**로 규명됨.
- **평점 흐름 진실:** 보정 전 2025년 여름 평점 폭락(1.71★~2.71★)은 5★ 과거 리뷰가 100개 제한에 잘려 나간 착시였으며, 절단 복원 평점(`trunc_corrected_rating`)은 전 기간 `4.00★~4.76★`로 안정적 흐름을 유지하되 **`2026-05` 자극 클레임 증가 시 `4.30★`로 실제 하락**했음을 포착함.

In [3]:
cols = ['year_month', 'raw_review_count', 'uncensored_1to4_count', 'recon_pop_total', 'ma3_recon_pop', 'raw_mean_rating', 'naive_ht_rating', 'trunc_corrected_rating']
print(monthly_df[monthly_df['year_month'] >= '2025-01'][cols].to_string(index=False))

year_month  raw_review_count  uncensored_1to4_count  recon_pop_total  ma3_recon_pop  raw_mean_rating  naive_ht_rating  trunc_corrected_rating
   2025-01                 3                      1            21.42          22.81            4.667            4.905                   4.756
   2025-02                 7                      7            93.79          52.75            2.714            3.230                   4.514
   2025-03                 9                      8            95.12          70.11            2.667            4.009                   4.466
   2025-04                 7                      7            84.78          91.23            2.429            3.005                   4.458
   2025-05                12                     12           171.24         117.05            2.917            3.336                   4.541
   2025-06                 7                      7            50.42         102.15            1.714            2.199                   4.004
   202

## 3. 전체 파이프라인 재현 실행
- `analysis.py`를 호출하여 정제 CSV 4종 및 시각화 차트 6종을 일괄 생성합니다.

In [4]:
import analysis
analysis.main()

정제 데이터셋 4종 및 절단 보정 시각화 차트 6종 생성 완료
